<a href="https://colab.research.google.com/github/tousifo/ml_notebooks/blob/main/eeg_qml_olvera_2024_safe_train_v5_strict_export.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Kaggle-ready official-code-aligned reproduction notebook

**Paper:** Cynthia Olvera, Oscar Montiel Ross, Yoshio Rubio, *EEG-based motor imagery classification with quantum algorithms*, Expert Systems With Applications 247 (2024) 123354.

This notebook is the corrected Kaggle version after auditing the authors' Code Ocean capsule. It prioritizes the paper's **subject-dependent cross-validation Hybrid EEGNet/DeepConvNet + VQC1** experiment because that path is implemented in the official `py_eeg_classifier.py`, `py_models.py`, and `q_classifier.py` files and is the cleanest first reproduction target.

## What changed after checking the official code

| Area | Previous reproduction risk | Official-code-aligned correction in this notebook |
|---|---|---|
| Dataset loader | Used a generic MOABB path by default. | Uses the authors' MATLAB-file logic: `B01T.mat/B01E.mat ... B09T.mat/B09E.mat`, when available. |
| MI crop | Used a cue-relative 4-second crop approximation. | Uses the official crop for dataset 2b: `t1 = 3.5 * 250`, `t2 = 7.5 * 250`, exactly 1000 samples. |
| Standardization | Generic channel z-score risk. | Uses the official `StandardScaler` behavior: fit on `X_train[:, 0, channel, :]`, transform train and test only with train-fitted scaler. |
| CV split | Generic subject split. | Mirrors official `train_test_split(..., test_size=0.2, random_state=42)` after merging T/E sessions, then 5-fold `StratifiedKFold(shuffle=False)` inside the remaining 80%. |
| VQC1 entanglement | Chain-only CNOTs. | Uses official even-odd entanglement: `(0→1), (2→3)` then `(1→2)` for 4 qubits. |
| Quantum layer | Worked conceptually but not capsule-faithful. | Reimplements official `DressedQuantumNet`: `Linear -> tanh -> π/2 -> VQC -> Linear`, with the official quantum dropout mask, but fixes CPU/GPU handling for Colab. |
| Classical CNNs | Approximate EEGNet/DCN variants. | Uses official PyTorch EEGNet and DeepConvNet layer choices, including EEGNet max pooling and separable convolution padding. |

**Strict note:** this notebook is faithful to the authors' released code where possible. It also fixes engineering fragility in the official code, especially the hardcoded CUDA device inside `DressedQuantumNet`, which breaks CPU-only Colab sessions.


### Safety patch in this version
This v5 copy keeps the dataset loading, split, model definitions, and evaluation logic intact, but fixes the dangerous no-training skip path from v3. The default run is a fresh B01 training run: one subject, one iteration, five folds, 150 epochs, no early stopping, corrected train metrics, persistent checkpoints, and visible training-start messages. It writes to a new v5 output directory so old completed-fold files cannot silently skip training. The final export cell now refuses to zip old `official_aligned` outputs and creates a run-specific zip named with the active `RUN_ID`. Use `START_POLICY = "resume"` only when you want to continue an interrupted v5 run.

## Paper/code reproduction target

Default target:

- Dataset: BCI Competition IV dataset 2b / BNCI 004-2014.
- Task: binary MI classification, left hand vs right hand.
- Subject-dependent cross-validation setup.
- Feature extractor: EEGNet or Deep ConvNet.
- Quantum classifier: VQC1.
- Qubits: 4.
- Quantum depth: 6.
- Optimizer: Adam.
- Learning rate: 0.001.
- LR decay: exponential factor 0.9 every 15 epochs.
- Batch size: official script default is 64; paper text says 128 for CV. The config cell exposes both. The default below uses **128** to match the paper text, but you can set `BATCH_SIZE = 64` to match the released script's parser default.
- Epochs: 300 in paper-scale mode; small value in debug mode.

The official code path is the strongest source of truth for hidden implementation details, but where paper text and parser defaults conflict, this notebook makes the choice explicit.


## 1. Install libraries

Run this first in Kaggle. Kaggle often already has most classical packages installed, but usually does **not** include PennyLane by default.

If package installation fails, turn **Internet = On** in the Kaggle notebook settings and rerun this cell. Do not bypass PennyLane for the hybrid-QNN experiment; that would no longer reproduce the paper's quantum classifier.


In [ ]:
# Kaggle-safe dependency setup.
# The cell installs only missing packages, so rerunning it is harmless.

import importlib.util  # Check whether a package is already installed.
import subprocess  # Run pip only when needed.
import sys  # Use the current notebook Python executable.

REQUIRED_PACKAGES = {  # Import name -> pip package name.
    "pennylane": "pennylane",  # Differentiable quantum circuit simulator.
    "torchinfo": "torchinfo",  # Optional model summary utility.
    "scipy": "scipy",  # Reads MATLAB .mat files.
    "sklearn": "scikit-learn",  # Splits, scaling, and metrics.
    "pandas": "pandas",  # Result tables.
    "matplotlib": "matplotlib",  # Training curves and confusion matrices.
}

missing = []  # Store pip packages that are not importable.
for import_name, pip_name in REQUIRED_PACKAGES.items():  # Check each dependency.
    if importlib.util.find_spec(import_name) is None:  # Package is missing.
        missing.append(pip_name)  # Mark it for installation.

if missing:  # Install only missing dependencies.
    print("Installing missing packages:", missing)  # Show what will be installed.
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + missing)  # Run pip.
else:
    print("All required packages are already installed.")  # No installation needed.


Installing missing packages: ['pennylane', 'torchinfo']


### Validation checklist

- **Matches paper/code?** Yes: the paper says Python and PennyLane were used, and the official capsule uses PyTorch for CV hybrid models.
- **Missing?** The paper does not publish exact package versions.
- **Approximation?** Version drift can change numerical behavior.
- **Data leakage risk?** None.
- **Reproducible?** Controlled later with seed setup.


## 2. Imports


In [ ]:
import os  # File paths and directory creation.
import gc  # Explicit garbage collection between folds.
import json  # Save metrics and configuration files.
import math  # Batch-loop and circuit utilities.
import random  # Python random seed control.
import time  # Runtime logging.
import copy  # Save best model weights per fold.
import warnings  # Silence non-critical warnings.
from pathlib import Path  # Safer filesystem paths.

import numpy as np  # Numerical arrays.
import pandas as pd  # Comparison tables.
import scipy.io as sio  # Read MATLAB .mat files used by the authors.

import torch  # PyTorch model training.
import torch.nn as nn  # Neural-network layers.
import torch.optim as optim  # Optimizers.
from torch.optim.lr_scheduler import ExponentialLR  # Official code uses ExponentialLR.

import pennylane as qml  # Quantum circuits.

import matplotlib.pyplot as plt  # Training curves and confusion matrices.
from IPython.display import display  # Render pandas tables in Kaggle notebooks.
from sklearn.model_selection import train_test_split, StratifiedKFold  # Official split logic.
from sklearn.preprocessing import StandardScaler  # Official per-channel standardization.
from sklearn.metrics import (  # Evaluation metrics used by the paper/code.
    accuracy_score,
    f1_score,
    cohen_kappa_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
)

warnings.filterwarnings("ignore")  # Keep notebook logs readable.


### Validation checklist

- **Matches paper/code?** Yes: PyTorch, PennyLane, scikit-learn, SciPy match the authors' implementation style.
- **Missing?** TensorFlow/Keras is not required for the CV hybrid PyTorch path.
- **Approximation?** None for the default CV reproduction path.
- **Data leakage risk?** None.
- **Reproducible?** Seed setup is next.


## 3. Reproducible seed setup


In [ ]:
BASE_SEED = 42  # Official split uses random_state=42; reuse it as the global base seed.


def set_all_seeds(seed: int = 42) -> None:
    """Set random seeds across Python, NumPy, PyTorch CPU, and PyTorch CUDA."""
    random.seed(seed)  # Fix Python's random module.
    np.random.seed(seed)  # Fix NumPy random generation.
    torch.manual_seed(seed)  # Fix PyTorch CPU random generation.
    torch.cuda.manual_seed_all(seed)  # Fix PyTorch CUDA random generation when GPU exists.
    torch.backends.cudnn.deterministic = True  # Prefer deterministic CUDA kernels.
    torch.backends.cudnn.benchmark = False  # Avoid nondeterministic kernel autotuning.


set_all_seeds(BASE_SEED)  # Apply the seed at notebook start.

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")  # Use GPU when Kaggle provides it.
print("Using device:", DEVICE)  # Confirm runtime device.


Using device: cpu


### Validation checklist

- **Matches paper/code?** The official code fixes the external split with `random_state=42` but does not fully seed all libraries.
- **What we implement:** A stricter controlled seed setup.
- **Approximation?** Small, beneficial reproducibility improvement.
- **Data leakage risk?** None.
- **Reproducible?** Yes, within library/hardware limits.


## 4. Experiment configuration


In [ ]:
# -------- Dataset paths --------
# The notebook first tries the Kaggle paths, then Colab paths. Section 5 can also extract /content/bci-iv-2b.zip.
KAGGLE_DATA_DIR = "/kaggle/input/datasets/tousifbnn/bci-iv-2b"  # Folder containing B01T.mat/B01E.mat ... B09T.mat/B09E.mat.
KAGGLE_FILE_HINT = "/kaggle/input/datasets/tousifbnn/bci-iv-2b/B01E.mat"  # Single-file hint you provided.

DATA_CANDIDATES = [  # Ordered paths to test.
    KAGGLE_DATA_DIR,  # Exact folder path you provided.
    str(Path(KAGGLE_FILE_HINT).parent),  # Parent folder of the example file.
    "/kaggle/input/bci-iv-2b",  # Common Kaggle mount variant.
    "/kaggle/input",  # Broad fallback search root.
    "/content/bci-iv-2b",  # Colab extraction folder used in Section 5.
    "/content/data/bci-iv-2b",  # Older Colab fallback.
    "/content",  # Broad Colab fallback search root.
]

EXPECTED_FILENAMES = [f"B0{s}{suffix}.mat" for s in range(1, 10) for suffix in ("T", "E")]  # 18 official MATLAB files.


def has_expected_files(folder: Path) -> bool:
    """Return True when a folder contains all 18 official MATLAB files."""
    return folder.exists() and folder.is_dir() and all((folder / name).exists() for name in EXPECTED_FILENAMES)


def resolve_data_dir(candidates: list[str]) -> Path:
    """Find the dataset folder from exact candidates, then limited recursive search."""
    for candidate in candidates:  # First try exact candidate folders.
        folder = Path(candidate)  # Convert string to Path.
        if has_expected_files(folder):  # Perfect match.
            return folder  # Use this folder.

    for candidate in candidates:  # Then search below candidate roots.
        root = Path(candidate)  # Convert to Path.
        if not root.exists():  # Skip missing roots.
            continue
        try:
            for hit in root.rglob("B01E.mat"):  # Search for one known file.
                folder = hit.parent  # Candidate dataset folder.
                if has_expected_files(folder):  # Verify all expected files exist.
                    return folder  # Use verified folder.
        except PermissionError:  # Some mounted paths may be protected.
            continue

    return Path(KAGGLE_DATA_DIR)  # Fall back to your requested path so the missing-file report is useful.


DATA_DIR = resolve_data_dir(DATA_CANDIDATES)  # Active dataset directory. Section 5 may update this after zip extraction.

# -------- Safe execution profile --------
# Keep the official data/model/split logic, but do not default to the full paper-scale job in Colab.
# Use "paper_scale" only after checkpoint/resume and B01 sanity checks are verified.
RUN_MODE = "colab_safe"  # Options: "quick_debug", "colab_safe", "paper_scale".

if RUN_MODE == "quick_debug":
    SUBJECTS = [0]  # B01 only.
    N_ITER = 1
    N_EPOCHS = 2
    K_FOLDS = 2
    EARLY_STOPPING = False
    PATIENCE = 2
    TRAIN_EVAL_INTERVAL = 0
elif RUN_MODE == "paper_scale":
    SUBJECTS = list(range(9))  # Official code uses zero-based subjects internally.
    N_ITER = 5  # Paper/code repeat five times for hybrid models.
    N_EPOCHS = 300  # Paper-scale CV training uses 300 epochs.
    K_FOLDS = 5
    EARLY_STOPPING = False  # Disabled by default to preserve paper-scale behavior.
    PATIENCE = 40
    TRAIN_EVAL_INTERVAL = 0  # Avoid extra compute in paper-scale runs.
elif RUN_MODE == "colab_safe":
    SUBJECTS = [0]  # B01 only. Change to [1], [2], ... after B01 is verified.
    N_ITER = 1
    N_EPOCHS = 150
    K_FOLDS = 5
    # Do NOT early-stop by default. The uploaded B01 run showed folds 1-3 stopped too early
    # before the delayed validation-accuracy improvements seen in the longer 300-epoch run.
    EARLY_STOPPING = False
    PATIENCE = 60
    TRAIN_EVAL_INTERVAL = 25  # Extra eval-mode train metrics only every 25 epochs.
else:
    raise ValueError(f"Unknown RUN_MODE: {RUN_MODE}")

DEBUG_MODE = RUN_MODE == "quick_debug"  # Backward-compatible flag for old notebook sections.
EARLY_STOP_MIN_DELTA = 0.0  # Minimum monitored-metric improvement required to reset patience.
EARLY_STOP_WARMUP_EPOCHS = 75  # Optional safety: never stop before this epoch when EARLY_STOPPING=True.
SAVE_CHECKPOINTS = True  # Save latest and best states so a disconnected run can resume.
RUN_SANITY_CHECKS = True  # Print/save label counts and sample predictions for each fold.

# -------- Rerun policy --------
# v3 could silently skip all folds because it reused old completed-fold files/checkpoints.
# v5 defaults to a NEW output directory and fresh training, so at least one batch must run.
RUN_ID = "metricfix_v5_b01_150ep"  # Change this only when you intentionally want a separate run folder.
START_POLICY = "fresh_train"  # Options: "fresh_train", "resume", "skip_completed".

if START_POLICY == "fresh_train":
    RESUME_TRAINING = False  # Do not load old checkpoints. Train from epoch 1.
    SKIP_COMPLETED_FOLDS = False  # Do not reuse old fold_result.json/model/pred files.
elif START_POLICY == "resume":
    RESUME_TRAINING = True  # Continue interrupted v4 latest checkpoints when epoch < N_EPOCHS.
    SKIP_COMPLETED_FOLDS = False  # Still train incomplete folds instead of silently trusting result files.
elif START_POLICY == "skip_completed":
    RESUME_TRAINING = True
    SKIP_COMPLETED_FOLDS = True  # Use only after you already verified a completed v4 run.
else:
    raise ValueError(f"Unknown START_POLICY: {START_POLICY}")

# -------- Model choice --------
# 1 = EEGNet, 2 = DeepConvNet, exactly as official py_eeg_classifier.py.
N_MODEL = 1  # Start with Hybrid EEGNet + VQC1, the clearest target in Table 4.
IS_HYBRID = True  # True replaces the classical classifier with DressedQuantumNet.

# -------- CV hyperparameters --------
BATCH_SIZE = 16 if DEBUG_MODE else 128  # Paper text: 128. Official parser default: 64.
LR = 1e-3  # Paper and official script default for CV.
DROPOUT = 0.5  # Official EEGNet/DCN dropout.
STEP_SIZE = 15  # Official LR decay interval.
GAMMA_LR = 0.9  # Official exponential LR decay factor.

# -------- Architecture parameters --------
N_CHANNELS = 3  # BCI IV-2b EEG channels used by the paper: C3, Cz, C4.
TIMEPOINTS = 1000  # Official crop: 3.5s to 7.5s at 250 Hz.
N_CLASSES = 2  # Left hand vs right hand.

# EEGNet paper/code settings.
EEGNET_KERNEL_LENGTH = 125  # Official CV default and paper Table 3.
EEGNET_POOL_SIZE = 5  # Official CV default; second pool is twice this value.
MAX_POOL = True  # Official parser default is true for EEGNet in CV.

# Deep ConvNet settings from paper Table 2 / official code.
DCN_KERNEL_LENGTH = 10
DCN_POOL_SIZE = 3
DCN_POOL_STRIDE = 3

# Quantum classifier settings.
N_QUBITS = 4  # Paper VQC1 uses 4 qubits.
Q_DEPTH = 6  # Paper VQC1 depth is 6.
Q_DELTA = 0.01  # Initial spread of random quantum weights.
Q_DROP = 0.01  # Official parser default for quantum dropout.
ANSATZ = 0  # Official ansatz 0 = VQC1 used in CV experiments.

# -------- Output/checkpoint paths --------
# Kaggle input is read-only; all outputs must go to /kaggle/working.
# In Colab, Drive is used when available so checkpoints survive runtime disconnects.
USE_GOOGLE_DRIVE_FOR_COLAB = True
IS_KAGGLE = Path("/kaggle/working").exists()

if IS_KAGGLE:
    RUN_ROOT = Path("/kaggle/working")
elif USE_GOOGLE_DRIVE_FOR_COLAB:
    try:
        from google.colab import drive  # Available only in Colab.
        drive.mount("/content/drive")
        RUN_ROOT = Path("/content/drive/MyDrive/eeg_qml_olvera_runs")
    except Exception as exc:
        print("Google Drive mount skipped/failed; checkpoints will be local to this runtime.")
        print("Reason:", repr(exc))
        RUN_ROOT = Path("/content")
else:
    RUN_ROOT = Path("/content")

OUTPUT_DIR = RUN_ROOT / f"eeg_qml_olvera_{RUN_ID}_outputs"  # Saved logs/results for this run ID.
CHECKPOINT_DIR = OUTPUT_DIR / "checkpoints"  # Fold-level latest/best checkpoints.
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)  # Create output folder.
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)  # Create checkpoint folder.

CONFIG = {  # Save only JSON-serializable configuration values.
    "KAGGLE_DATA_DIR": KAGGLE_DATA_DIR,
    "KAGGLE_FILE_HINT": KAGGLE_FILE_HINT,
    "DATA_DIR": str(DATA_DIR),
    "RUN_MODE": RUN_MODE,
    "RUN_ID": RUN_ID,
    "START_POLICY": START_POLICY,
    "DEBUG_MODE": DEBUG_MODE,
    "SUBJECTS": SUBJECTS,
    "N_ITER": N_ITER,
    "N_EPOCHS": N_EPOCHS,
    "K_FOLDS": K_FOLDS,
    "EARLY_STOPPING": EARLY_STOPPING,
    "PATIENCE": PATIENCE,
    "EARLY_STOP_MIN_DELTA": EARLY_STOP_MIN_DELTA,
    "SAVE_CHECKPOINTS": SAVE_CHECKPOINTS,
    "RESUME_TRAINING": RESUME_TRAINING,
    "SKIP_COMPLETED_FOLDS": SKIP_COMPLETED_FOLDS,
    "RUN_SANITY_CHECKS": RUN_SANITY_CHECKS,
    "TRAIN_EVAL_INTERVAL": TRAIN_EVAL_INTERVAL,
    "N_MODEL": N_MODEL,
    "IS_HYBRID": IS_HYBRID,
    "BATCH_SIZE": BATCH_SIZE,
    "LR": LR,
    "DROPOUT": DROPOUT,
    "STEP_SIZE": STEP_SIZE,
    "GAMMA_LR": GAMMA_LR,
    "N_CHANNELS": N_CHANNELS,
    "TIMEPOINTS": TIMEPOINTS,
    "N_CLASSES": N_CLASSES,
    "EEGNET_KERNEL_LENGTH": EEGNET_KERNEL_LENGTH,
    "EEGNET_POOL_SIZE": EEGNET_POOL_SIZE,
    "MAX_POOL": MAX_POOL,
    "DCN_KERNEL_LENGTH": DCN_KERNEL_LENGTH,
    "DCN_POOL_SIZE": DCN_POOL_SIZE,
    "DCN_POOL_STRIDE": DCN_POOL_STRIDE,
    "N_QUBITS": N_QUBITS,
    "Q_DEPTH": Q_DEPTH,
    "Q_DELTA": Q_DELTA,
    "Q_DROP": Q_DROP,
    "ANSATZ": ANSATZ,
    "OUTPUT_DIR": str(OUTPUT_DIR),
    "CHECKPOINT_DIR": str(CHECKPOINT_DIR),
}
with open(OUTPUT_DIR / "config.json", "w") as f:
    json.dump(CONFIG, f, indent=2)  # Save run configuration.

print(json.dumps(CONFIG, indent=2))  # Show active config.
print("Resolved DATA_DIR:", DATA_DIR)  # Confirm active data path.
print("Output directory:", OUTPUT_DIR)  # Confirm writable output path.
print("Checkpoint directory:", CHECKPOINT_DIR)  # Confirm persistent checkpoint path.
print("START_POLICY:", START_POLICY, "| RESUME_TRAINING:", RESUME_TRAINING, "| SKIP_COMPLETED_FOLDS:", SKIP_COMPLETED_FOLDS)
if START_POLICY == "fresh_train":
    print("Fresh-training mode active: old v3/v4 completed-fold files will NOT be reused.")


Google Drive mount skipped/failed; checkpoints will be local to this runtime.
Reason: ValueError('mount failed')
{
  "KAGGLE_DATA_DIR": "/kaggle/input/datasets/tousifbnn/bci-iv-2b",
  "KAGGLE_FILE_HINT": "/kaggle/input/datasets/tousifbnn/bci-iv-2b/B01E.mat",
  "DATA_DIR": "/kaggle/input/datasets/tousifbnn/bci-iv-2b",
  "RUN_MODE": "colab_safe",
  "RUN_ID": "metricfix_v5_b01_150ep",
  "START_POLICY": "fresh_train",
  "DEBUG_MODE": false,
  "SUBJECTS": [
    0
  ],
  "N_ITER": 1,
  "N_EPOCHS": 150,
  "K_FOLDS": 5,
  "EARLY_STOPPING": false,
  "PATIENCE": 60,
  "EARLY_STOP_MIN_DELTA": 0.0,
  "SAVE_CHECKPOINTS": true,
  "RESUME_TRAINING": false,
  "SKIP_COMPLETED_FOLDS": false,
  "RUN_SANITY_CHECKS": true,
  "TRAIN_EVAL_INTERVAL": 25,
  "N_MODEL": 1,
  "IS_HYBRID": true,
  "BATCH_SIZE": 128,
  "LR": 0.001,
  "DROPOUT": 0.5,
  "STEP_SIZE": 15,
  "GAMMA_LR": 0.9,
  "N_CHANNELS": 3,
  "TIMEPOINTS": 1000,
  "N_CLASSES": 2,
  "EEGNET_KERNEL_LENGTH": 125,
  "EEGNET_POOL_SIZE": 5,
  "MAX_POOL": t

### Validation checklist

- **Matches paper/code?** Full mode matches the core CV experiment. Debug mode is intentionally smaller.
- **Kaggle path fixed?** Yes: the notebook now defaults to `/kaggle/input/datasets/tousifbnn/bci-iv-2b` and writes outputs to `/kaggle/working`.
- **Paper-code conflict:** The paper text says batch size 128 for CV; the released script parser default is 64. This notebook defaults to 128 in full mode and documents the conflict.
- **Approximation?** Debug mode is not a reproduction result.
- **Data leakage risk?** None.
- **Reproducible?** Yes; config is saved.


## 5. Dataset setup instructions for Kaggle

This Kaggle version expects the MATLAB files in the folder you provided:

```text
/kaggle/input/datasets/tousifbnn/bci-iv-2b/B01T.mat
/kaggle/input/datasets/tousifbnn/bci-iv-2b/B01E.mat
...
/kaggle/input/datasets/tousifbnn/bci-iv-2b/B09T.mat
/kaggle/input/datasets/tousifbnn/bci-iv-2b/B09E.mat
```

Kaggle `/kaggle/input` is read-only. Results, plots, models, and CSV files are saved under:

```text
/kaggle/working/eeg_qml_olvera_official_aligned_outputs
```

Do **not** move outputs into `/kaggle/input`; Kaggle will reject writes there.


In [ ]:
# Kaggle dataset sanity check.
# This prints the active folder and a short listing to catch path mistakes early.

print("Active DATA_DIR:", DATA_DIR)  # Show the resolved dataset directory.
print("DATA_DIR exists:", DATA_DIR.exists())  # Confirm path exists.
if DATA_DIR.exists():  # List a few files for inspection.
    print("First files:", sorted([p.name for p in DATA_DIR.iterdir()])[:20])  # Show filenames.
else:
    print("Kaggle input roots:")  # Help debug wrong dataset mount.
    input_root = Path("/kaggle/input")  # Kaggle read-only input root.
    if input_root.exists():  # Only list if available.
        print(sorted([p.as_posix() for p in input_root.iterdir()])[:20])  # Show mounted datasets.


Active DATA_DIR: /kaggle/input/datasets/tousifbnn/bci-iv-2b
DATA_DIR exists: False
Kaggle input roots:
[]


In [ ]:
from pathlib import Path
import zipfile

COLAB_ZIP_PATH = Path("/content/bci-iv-2b.zip")
COLAB_EXTRACT_ROOT = Path("/content/bci-iv-2b")

EXPECTED_FILENAMES = [f"B0{s}{suffix}.mat" for s in range(1, 10) for suffix in ("T", "E")]

def has_expected_files(folder: Path) -> bool:
    return folder.exists() and folder.is_dir() and all((folder / name).exists() for name in EXPECTED_FILENAMES)

def find_dataset_folder(root: Path):
    if not root.exists():
        return None
    if has_expected_files(root):
        return root
    for hit in root.rglob("B01E.mat"):
        folder = hit.parent
        if has_expected_files(folder):
            return folder
    return None

if COLAB_ZIP_PATH.exists() and find_dataset_folder(COLAB_EXTRACT_ROOT) is None:
    COLAB_EXTRACT_ROOT.mkdir(parents=True, exist_ok=True)
    print(f"Extracting {COLAB_ZIP_PATH} to {COLAB_EXTRACT_ROOT} ...")
    with zipfile.ZipFile(COLAB_ZIP_PATH, "r") as zf:
        zf.extractall(COLAB_EXTRACT_ROOT)

DATA_DIR = find_dataset_folder(COLAB_EXTRACT_ROOT) or find_dataset_folder(Path("/content")) or find_dataset_folder(Path("/kaggle/input")) or DATA_DIR
missing_files = [DATA_DIR / name for name in EXPECTED_FILENAMES if not (DATA_DIR / name).exists()]

# Keep the saved config aligned with the final resolved data folder after optional Colab extraction.
CONFIG["DATA_DIR"] = str(DATA_DIR)
with open(OUTPUT_DIR / "config.json", "w") as f:
    json.dump(CONFIG, f, indent=2)

print("Resolved DATA_DIR:", DATA_DIR)
print("Missing files:", len(missing_files))
if missing_files:
    print("Examples:", [str(p) for p in missing_files[:10]])

assert DATA_DIR.exists(), "DATA_DIR does not exist after extracting /content/bci-iv-2b.zip."
assert len(missing_files) == 0, "Not all BCI IV-2b .mat files were found."


Extracting /content/bci-iv-2b.zip to /content/bci-iv-2b ...
Resolved DATA_DIR: /content/bci-iv-2b
Missing files: 0


### Validation checklist

- **Matches paper/code?** Yes for the faithful MATLAB route.
- **Missing?** If files are missing, training cannot start. Do not substitute random or toy data.
- **Approximation?** None in the MATLAB route.
- **Data leakage risk?** None.
- **Reproducible?** Yes when the same MATLAB files are used.


## 6. Official-code-aligned MATLAB data loader


In [ ]:
def load_data_mat(data_path: Path, subject: int, training: bool) -> tuple[np.ndarray, np.ndarray]:
    """Load full 8-second trials exactly like the authors' get_trials.load_data for dataset 2b.

    Parameters
    ----------
    data_path:
        Folder containing B01T.mat/B01E.mat ... B09T.mat/B09E.mat.
    subject:
        One-based subject index in [1, 9].
    training:
        True loads B0xT.mat; False loads B0xE.mat.

    Returns
    -------
    X:
        Array with shape (trials, 3, 2000), before the 3.5s--7.5s crop.
    y:
        Original labels from MATLAB files, normally 1=left hand, 2=right hand.
    """
    n_channels = 3  # Official dataset 2b EEG channels used by the paper.
    max_trials = 440  # Official preallocation size for dataset 2b.
    window_length = 8 * 250  # Official full trial length before MI-period crop.

    suffix = "T" if training else "E"  # T=session/train file, E=evaluation file.
    mat_file = data_path / f"B0{subject}{suffix}.mat"  # Official filename pattern.
    if not mat_file.exists():  # Fail loudly if the dataset is missing.
        raise FileNotFoundError(f"Missing {mat_file}. Put BCI IV-2b MATLAB files in DATA_DIR.")

    mat = sio.loadmat(mat_file)  # Load MATLAB file.
    mat_data = mat["data"]  # Official files store runs in a MATLAB struct array named 'data'.

    X_out = np.zeros((max_trials, n_channels, window_length), dtype=np.float32)  # Preallocate EEG trials.
    y_out = np.zeros(max_trials, dtype=np.int64)  # Preallocate labels.
    n_valid = 0  # Count valid trials copied into arrays.

    for run_idx in range(mat_data.size):  # Iterate over runs inside the MATLAB struct.
        run = mat_data[0, run_idx][0, 0]  # Unwrap MATLAB cell/struct nesting.
        X_cont = run[0]  # Continuous EEG/EOG array, time x channels.
        trial_starts = run[1].squeeze()  # Trial start sample indices.
        labels = run[2].squeeze()  # Class labels.
        artifacts = run[5].squeeze()  # Artifact flags; official default keeps all trials.

        for trial_idx, start in enumerate(trial_starts):  # Iterate trials in this run.
            start = int(start)  # Convert MATLAB index value to Python int.
            stop = start + window_length  # End sample for the 8-second trial.
            if stop > X_cont.shape[0]:  # Guard against malformed files.
                continue
            X_out[n_valid] = X_cont[start:stop, :n_channels].T  # Convert time x channels to channels x time.
            y_out[n_valid] = int(labels[trial_idx])  # Store original label 1 or 2.
            n_valid += 1  # Increment valid-trial counter.

    return X_out[:n_valid], y_out[:n_valid]


def standardize_data_official(X_train: np.ndarray, X_test: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    """Apply the official per-channel StandardScaler logic without test leakage."""
    X_train = X_train.copy()  # Avoid modifying caller arrays in place.
    X_test = X_test.copy()  # Avoid modifying caller arrays in place.
    n_channels = X_train.shape[2]  # X shape is trials x 1 x channels x time.

    for ch in range(n_channels):  # Fit one scaler per EEG channel.
        scaler = StandardScaler()  # Official scikit-learn scaler.
        scaler.fit(X_train[:, 0, ch, :])  # Fit only on training trials for this channel.
        X_train[:, 0, ch, :] = scaler.transform(X_train[:, 0, ch, :])  # Transform training data.
        X_test[:, 0, ch, :] = scaler.transform(X_test[:, 0, ch, :])  # Transform test data with train scaler.

    return X_train, X_test


def get_data_official(data_path: Path, subject_zero_based: int, standardize: bool = True) -> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    """Load and crop one subject using the official get_trials.get_data logic for dataset 2b."""
    fs = 250  # Dataset sampling rate.
    t1 = int(3.5 * fs)  # Official crop start for dataset 2b.
    t2 = int(7.5 * fs)  # Official crop end for dataset 2b.
    T = t2 - t1  # Must be 1000 samples.

    subject_one_based = subject_zero_based + 1  # Official public filenames are B01..B09.
    X_train_raw, y_train = load_data_mat(data_path, subject_one_based, training=True)  # Load B0xT.
    X_test_raw, y_test = load_data_mat(data_path, subject_one_based, training=False)  # Load B0xE.

    n_train = X_train_raw.shape[0]  # Number of T-session trials.
    n_test = X_test_raw.shape[0]  # Number of E-session trials.
    n_channels = X_train_raw.shape[1]  # Should be 3.

    X_train = X_train_raw[:, :, t1:t2].reshape(n_train, 1, n_channels, T)  # Crop and add Conv2D input axis.
    X_test = X_test_raw[:, :, t1:t2].reshape(n_test, 1, n_channels, T)  # Crop and add Conv2D input axis.

    if standardize:  # Match official default.
        X_train, X_test = standardize_data_official(X_train, X_test)  # Fit scaler on train only.

    y_train = (y_train - 1).astype(np.int64)  # Map labels 1/2 to 0/1.
    y_test = (y_test - 1).astype(np.int64)  # Map labels 1/2 to 0/1.

    return X_train.astype(np.float32), y_train, X_test.astype(np.float32), y_test


In [ ]:
def load_subject_cv_pool(subject_zero_based: int) -> tuple[np.ndarray, np.ndarray]:
    """Merge T and E sessions for the official subject-dependent CV experiment."""
    X_train, y_train, X_test, y_test = get_data_official(DATA_DIR, subject_zero_based, standardize=True)  # Load official split.
    X_all = np.append(X_train, X_test, axis=0)  # Official CV code merges T and E sessions.
    y_all = np.append(y_train, y_test, axis=0).astype(np.int64)  # Merge labels.
    return X_all, y_all


# Smoke-test the data loader only if files are present.
if len(missing_files) == 0:
    X_smoke, y_smoke = load_subject_cv_pool(SUBJECTS[0])  # Load first configured subject.
    print("Loaded shape:", X_smoke.shape)  # Expected: trials x 1 x 3 x 1000.
    print("Labels:", dict(zip(*np.unique(y_smoke, return_counts=True))))  # Inspect class balance.
    assert X_smoke.ndim == 4 and X_smoke.shape[1:] == (1, 3, 1000)  # Guard expected tensor shape.
    assert set(np.unique(y_smoke)).issubset({0, 1})  # Guard binary label mapping.
else:
    print("Dataset files are missing, so loader smoke test is skipped.")


Loaded shape: (720, 1, 3, 1000)
Labels: {np.int64(0): np.int64(360), np.int64(1): np.int64(360)}


### Validation checklist

- **Matches paper/code?** Yes: this mirrors `get_trials.py` for dataset 2b.
- **Paper/code support:** official code crops `3.5s–7.5s`, returns `(trials, 1, 3, 1000)`, and uses train-fitted `StandardScaler`.
- **Approximation?** None if using the same MATLAB files.
- **Data leakage risk?** Low: scaler is fit on T-session data before merging for CV. This matches code, but it is a methodological blind spot because CV later merges T/E; a stricter variant would fit scalers inside each fold.
- **Reproducible?** Yes.


## 7. Official PyTorch EEGNet and DeepConvNet


In [ ]:
class SeparableConv2d(nn.Module):
    """Official separable convolution block from py_models.py."""

    def __init__(self, in_channels, out_channels, kernel_size, stride=1, padding=0, dilation=1, bias=False):
        super().__init__()  # Initialize parent module.
        self.depthwise = nn.Conv2d(  # Depthwise temporal convolution.
            in_channels,
            in_channels,
            kernel_size,
            stride,
            padding,
            dilation,
            groups=in_channels,
            bias=False,
        )
        self.pointwise = nn.Conv2d(in_channels, out_channels, 1, 1, 0, 1, 1, bias=bias)  # Pointwise channel mixing.

    def forward(self, x):
        x = self.depthwise(x)  # Apply independent depthwise filters.
        x = self.pointwise(x)  # Combine feature maps with 1x1 convolution.
        return x  # Return separable convolution output.


class EEGNetOfficial(nn.Module):
    """Official-code-aligned EEGNet feature extractor plus replaceable classifier."""

    def __init__(
        self,
        n_classes=2,
        n_channels=3,
        timepoints=1000,
        dropout_rate=0.5,
        F1=8,
        D=2,
        F2=16,
        kernel_length=125,
        pool_ks=(5, 10),
        max_pool=True,
        dense=False,
    ):
        super().__init__()  # Initialize parent module.
        self.isDense = dense  # Official switch for alternate dense classifier.
        pooling = nn.MaxPool2d if max_pool else nn.AvgPool2d  # Official CV default uses max pooling.

        self.block1 = nn.Sequential(
            nn.Conv2d(1, F1, (1, kernel_length), padding=(0, kernel_length // 2), bias=False),  # Temporal conv.
            nn.BatchNorm2d(F1),  # Normalize temporal feature maps.
            nn.Conv2d(F1, F1 * D, (n_channels, 1), padding=0, groups=F1, bias=False),  # Depthwise spatial conv.
            nn.BatchNorm2d(F1 * D),  # Normalize spatially filtered maps.
            nn.ELU(),  # Nonlinear activation used by EEGNet.
            pooling(kernel_size=(1, pool_ks[0])),  # First temporal downsampling.
            nn.Dropout(dropout_rate),  # Regularization.
        )

        self.block2 = nn.Sequential(
            SeparableConv2d(F1 * D, F2, kernel_size=(1, 16), padding=(0, 8), bias=False),  # Separable conv.
            nn.BatchNorm2d(F2),  # Normalize selected features.
            nn.ELU(),  # Nonlinear activation.
            pooling(kernel_size=(1, pool_ks[1])),  # Second temporal downsampling.
            nn.Dropout(dropout_rate),  # Regularization.
        )

        self.flatten = nn.Flatten()  # Convert feature maps to a vector.
        feat_size = F2 * int(timepoints / (pool_ks[0] * pool_ks[1]))  # Official feature size formula.
        self.feat_size = feat_size  # Store feature size for the quantum classifier.
        self.classifier = nn.Linear(feat_size, n_classes)  # Classical classifier placeholder.

    def forward(self, x):
        x = self.block1(x)  # Extract temporal and spatial EEG features.
        x = self.block2(x)  # Extract/choose higher-level features.
        x = self.flatten(x)  # Flatten to vector.
        x = self.classifier(x)  # Classify with classical or quantum replacement layer.
        return x  # Return logits.


class DeepConvNetOfficial(nn.Module):
    """Official-code-aligned Deep ConvNet feature extractor plus replaceable classifier."""

    def __init__(
        self,
        n_classes=2,
        n_channels=3,
        dropout_rate=0.5,
        kernel_length=10,
        pool_ks=3,
        pool_stride=3,
        feat_size=1400,
    ):
        super().__init__()  # Initialize parent module.
        self.feat_size = feat_size  # Store expected flatten size.

        self.block1 = nn.Sequential(
            nn.Conv2d(1, 25, kernel_size=(1, kernel_length), stride=(1, 1)),  # Temporal conv.
            nn.Conv2d(25, 25, kernel_size=(n_channels, 1), stride=(1, 1)),  # Spatial conv.
            nn.BatchNorm2d(25, affine=False),  # Official affine=False.
            nn.ELU(),  # Activation.
            nn.MaxPool2d(kernel_size=(1, pool_ks), stride=(1, pool_stride)),  # Temporal max pooling.
            nn.Dropout(dropout_rate),  # Regularization.
        )
        self.block2 = nn.Sequential(
            nn.Conv2d(25, 50, kernel_size=(1, kernel_length), stride=(1, 1)),  # Conv block 2.
            nn.BatchNorm2d(50, affine=False),
            nn.ELU(),
            nn.MaxPool2d(kernel_size=(1, pool_ks), stride=(1, pool_stride)),
            nn.Dropout(dropout_rate),
        )
        self.block3 = nn.Sequential(
            nn.Conv2d(50, 100, kernel_size=(1, kernel_length), stride=(1, 1)),  # Conv block 3.
            nn.BatchNorm2d(100, affine=False),
            nn.ELU(),
            nn.MaxPool2d(kernel_size=(1, pool_ks), stride=(1, pool_stride)),
            nn.Dropout(dropout_rate),
        )
        self.block4 = nn.Sequential(
            nn.Conv2d(100, 200, kernel_size=(1, kernel_length), stride=(1, 1)),  # Conv block 4.
            nn.BatchNorm2d(200, affine=False),
            nn.ELU(),
            nn.MaxPool2d(kernel_size=(1, pool_ks), stride=(1, pool_stride)),
            nn.Dropout(dropout_rate),
        )
        self.flatten = nn.Flatten()  # Flatten feature maps.
        self.classifier = nn.Linear(feat_size, n_classes)  # Classical classifier placeholder.

    def forward(self, x):
        x = self.block1(x)  # First temporal+spatial block.
        x = self.block2(x)  # Second conv block.
        x = self.block3(x)  # Third conv block.
        x = self.block4(x)  # Fourth conv block.
        x = self.flatten(x)  # Flatten to vector.
        x = self.classifier(x)  # Classify with classical or quantum replacement layer.
        return x  # Return logits.


In [ ]:
def infer_flatten_size(feature_model: nn.Module, input_shape=(1, 1, 3, 1000)) -> int:
    """Infer flatten size by running the feature extractor up to flatten."""
    feature_model.eval()  # Disable dropout for deterministic shape inference.
    with torch.no_grad():  # No gradients needed for shape inference.
        x = torch.zeros(input_shape, dtype=torch.float32)  # Dummy EEG tensor.
        if isinstance(feature_model, EEGNetOfficial):
            x = feature_model.block1(x)  # EEGNet block 1.
            x = feature_model.block2(x)  # EEGNet block 2.
            x = feature_model.flatten(x)  # Flatten features.
        else:
            x = feature_model.block1(x)  # DCN block 1.
            x = feature_model.block2(x)  # DCN block 2.
            x = feature_model.block3(x)  # DCN block 3.
            x = feature_model.block4(x)  # DCN block 4.
            x = feature_model.flatten(x)  # Flatten features.
    return int(x.shape[1])  # Return feature dimension.


# Validate model feature sizes on dummy input.
_eegnet = EEGNetOfficial(kernel_length=EEGNET_KERNEL_LENGTH, pool_ks=(EEGNET_POOL_SIZE, EEGNET_POOL_SIZE * 2))
_dcn = DeepConvNetOfficial(kernel_length=DCN_KERNEL_LENGTH, pool_ks=DCN_POOL_SIZE, pool_stride=DCN_POOL_STRIDE)
print("EEGNet declared/inferred feat size:", _eegnet.feat_size, infer_flatten_size(_eegnet))
print("DCN declared/inferred feat size:", _dcn.feat_size, infer_flatten_size(_dcn))


EEGNet declared/inferred feat size: 320 320
DCN declared/inferred feat size: 1400 1400


### Validation checklist

- **Matches paper/code?** Yes for the official PyTorch CV model path.
- **Paper support:** Table 3 gives the EEGNet/VQC1 block; Table 2 gives the Deep ConvNet/VQC1 block.
- **Official-code detail:** EEGNet uses max pooling in the released script, even though the original EEGNet paper used average pooling.
- **Approximation?** Classical classifier here returns logits rather than Softmax probabilities to work correctly with `CrossEntropyLoss`. Hybrid path is unaffected because the quantum layer returns logits through its final linear layer.
- **Data leakage risk?** None.
- **Reproducible?** Yes.


## 8. Official VQC1 and dressed quantum classifier


In [ ]:
class VQC1Official:
    """VQC1 from official q_classifier.py, ansatz 0."""

    def __init__(self, n_qubits=4, q_depth=6):
        self.n_qubits = n_qubits  # Number of quantum wires.
        self.q_depth = q_depth  # Number of trainable variational layers.
        self.dev = qml.device("default.qubit", wires=self.n_qubits)  # CPU-based PennyLane simulator.
        self.qnode = qml.QNode(self.quantum_net, self.dev, interface="torch")  # Differentiable torch QNode.

    def H_layer(self):
        """Apply Hadamard gates to create an unbiased |+> state."""
        for idx in range(self.n_qubits):  # Loop over all qubits.
            qml.Hadamard(wires=idx)  # Put qubit into superposition.

    def RY_layer(self, weights):
        """Apply one RY rotation to each qubit."""
        for idx, angle in enumerate(weights):  # Pair each angle with a wire.
            qml.RY(angle, wires=idx)  # Rotate around the Y axis.

    def entangling_layer(self):
        """Official even-odd CNOT entangling layer."""
        for i in range(0, self.n_qubits - 1, 2):  # Even controls: 0->1, 2->3.
            qml.CNOT(wires=[i, i + 1])  # Apply CNOT.
        for i in range(1, self.n_qubits - 1, 2):  # Odd controls: 1->2.
            qml.CNOT(wires=[i, i + 1])  # Apply CNOT.

    def quantum_net(self, q_input_features, q_weights_flat):
        """Forward pass through VQC1."""
        q_weights = q_weights_flat.reshape(self.q_depth, self.n_qubits)  # Reshape trainable angles.
        self.H_layer()  # Initialize all qubits in superposition.
        self.RY_layer(q_input_features)  # Angle-encode 4 preprocessed EEG features.
        for layer_idx in range(self.q_depth):  # Repeat variational layers.
            self.entangling_layer()  # Apply fixed CNOT pattern.
            self.RY_layer(q_weights[layer_idx])  # Apply trainable RY layer.
        return tuple(qml.expval(qml.PauliZ(i)) for i in range(self.n_qubits))  # Measure Pauli-Z per qubit.


class DressedQuantumNetOfficialFixed(nn.Module):
    """Official dressed quantum classifier with Kaggle CUDA-safe PennyLane handling.

    Why this class is careful about devices:
    - The CNN can run on CUDA.
    - PennyLane's default.qubit simulator runs on CPU tensors.
    - Passing CUDA tensors directly into default.qubit causes the runtime error:
      "Expected all tensors to be on the same device, but found cuda:0 and cpu".
    - We therefore copy only the 4 quantum inputs and 24 quantum parameters to CPU,
      run the QNode, then copy the 4 measurement outputs back to the model device.
    - The copy operations remain differentiable in PyTorch, so gradients still flow
      back through the pre-net, quantum parameters, and post-net.
    """

    def __init__(self, in_size, n_qubits=4, q_depth=6, n_classes=2, q_delta=0.01, q_drop=0.01):
        super().__init__()  # Initialize parent module.
        self.n_qubits = n_qubits  # Number of qubits.
        self.q_depth = q_depth  # Number of quantum layers.
        self.q_drop = q_drop  # Official quantum dropout probability.
        self.pre_net = nn.Linear(in_size, n_qubits)  # Classical preprocessing layer: features -> qubits.
        self.q_params = nn.Parameter(q_delta * torch.randn(q_depth * n_qubits))  # Trainable quantum angles.
        self.post_net = nn.Linear(n_qubits, n_classes)  # Classical postprocessing: measurements -> class logits.
        self.vqc = VQC1Official(n_qubits=n_qubits, q_depth=q_depth)  # VQC1 circuit.
        self.qnode_device = torch.device("cpu")  # default.qubit is CPU-backed; keep QNode tensors on CPU.

    def _quantum_dropout_params(self):
        """Apply official Bernoulli mask to quantum weights during training."""
        if self.training and self.q_drop > 0:  # Only mask during training.
            keep_prob = 1.0 - self.q_drop  # Probability of keeping a quantum parameter.
            mask = torch.bernoulli(torch.full_like(self.q_params, keep_prob))  # Sample mask on same device/dtype.
            return self.q_params * mask  # Return masked quantum weights.
        return self.q_params  # Evaluation uses all quantum weights.

    def forward(self, input_features):
        model_device = input_features.device  # Remember whether the surrounding CNN is on CPU or CUDA.
        model_dtype = input_features.dtype  # Preserve the surrounding model dtype.

        pre_out = self.pre_net(input_features)  # Reduce CNN feature vector to n_qubits values.
        q_in = torch.tanh(pre_out) * (torch.pi / 2.0)  # Official tanh scaling before VQC.
        q_params = self._quantum_dropout_params()  # Apply official quantum dropout behavior.

        q_in_cpu = q_in.to(self.qnode_device)  # Move only 4 values/sample to CPU for default.qubit.
        q_params_cpu = q_params.to(self.qnode_device)  # Move only 24 trainable angles to CPU for default.qubit.

        q_outputs = []  # Collect one VQC output per sample on CPU.
        for elem_cpu in q_in_cpu:  # PennyLane QNode processes one sample at a time.
            q_res = self.vqc.qnode(elem_cpu, q_params_cpu)  # Evaluate differentiable quantum circuit on CPU.
            if isinstance(q_res, (tuple, list)):  # New PennyLane may return tuple of scalar tensors.
                q_res = torch.stack(list(q_res))  # Convert tuple to tensor shape (n_qubits,).
            q_outputs.append(q_res.to(dtype=model_dtype))  # Keep CPU result but match model dtype.

        q_out_cpu = torch.stack(q_outputs, dim=0)  # CPU batch shape: batch x n_qubits.
        q_out = q_out_cpu.to(model_device)  # Move quantum measurements back to CNN/post-net device.
        logits = self.post_net(q_out)  # Map quantum measurements to two class logits.
        return logits  # Return logits for CrossEntropyLoss.

### Kaggle CUDA/PennyLane device note

The hybrid model can keep the CNN on CUDA, but `default.qubit` is CPU-backed. The dressed QNN cell above moves only the tiny quantum tensors to CPU for the QNode call, then moves the 4 measurement outputs back to the active PyTorch device. This fixes the Kaggle error `Expected all tensors to be on the same device, but found cuda:0 and cpu`.

In [ ]:
def build_model(n_model: int = 1, is_hybrid: bool = True) -> nn.Module:
    """Build EEGNet or DeepConvNet and optionally replace its classifier with VQC1."""
    if n_model == 1:  # EEGNet.
        pool_ks = (EEGNET_POOL_SIZE, EEGNET_POOL_SIZE * 2)  # Official EEGNet pool sizes.
        model = EEGNetOfficial(
            n_classes=N_CLASSES,
            n_channels=N_CHANNELS,
            timepoints=TIMEPOINTS,
            dropout_rate=DROPOUT,
            kernel_length=EEGNET_KERNEL_LENGTH,
            pool_ks=pool_ks,
            max_pool=MAX_POOL,
        )
        in_size = model.feat_size  # Official formula gives 320 for EEGNet.
    elif n_model == 2:  # DeepConvNet.
        model = DeepConvNetOfficial(
            n_classes=N_CLASSES,
            n_channels=N_CHANNELS,
            dropout_rate=DROPOUT,
            kernel_length=DCN_KERNEL_LENGTH,
            pool_ks=DCN_POOL_SIZE,
            pool_stride=DCN_POOL_STRIDE,
            feat_size=1400,
        )
        in_size = model.feat_size  # Official expected size is 1400.
    else:
        raise ValueError("n_model must be 1 for EEGNet or 2 for DeepConvNet.")

    if is_hybrid:  # Replace final classical classifier with dressed quantum classifier.
        model.classifier = DressedQuantumNetOfficialFixed(
            in_size=in_size,
            n_qubits=N_QUBITS,
            q_depth=Q_DEPTH,
            n_classes=N_CLASSES,
            q_delta=Q_DELTA,
            q_drop=Q_DROP,
        )
    return model  # Return complete model.


# Quantum forward/backward smoke test on dummy data.
set_all_seeds(BASE_SEED)  # Reset seed before model creation.
smoke_model = build_model(N_MODEL, IS_HYBRID).to(DEVICE)  # Build configured model.
smoke_x = torch.randn(2, 1, 3, 1000, device=DEVICE)  # Small dummy EEG batch.
smoke_y = torch.tensor([0, 1], device=DEVICE)  # Dummy labels.
smoke_logits = smoke_model(smoke_x)  # Forward pass through CNN + QNN.
smoke_loss = nn.CrossEntropyLoss()(smoke_logits, smoke_y)  # Compute loss.
smoke_loss.backward()  # Backprop through classical and quantum parameters.
print("Smoke logits shape:", tuple(smoke_logits.shape))  # Expected (2, 2).
print("Smoke loss:", float(smoke_loss.detach().cpu()))  # Confirm numerical loss.
del smoke_model, smoke_x, smoke_y, smoke_logits, smoke_loss  # Free memory.
gc.collect()  # Collect Python objects.


Smoke logits shape: (2, 2)
Smoke loss: 1.0194642543792725


592

### Validation checklist

- **Matches paper/code?** Yes: VQC1 is `H -> RY(input) -> [CNOT even/odd + RY(weights)] × depth -> Pauli-Z expectations`, followed by a classical linear layer.
- **Paper support:** The VQC1 diagram and description specify 4 qubits, depth 6, RY rotations, CNOT entanglement, and Pauli-Z measurement.
- **Official-code fix:** The original code hardcodes `torch.device("cuda")` inside the quantum layer. This notebook fixes that so CPU-only Colab does not crash.
- **Approximation?** Device fix should not change the algorithm.
- **Data leakage risk?** None.
- **Reproducible?** Yes, subject to quantum simulator/library determinism.


## 9. Training and evaluation utilities


In [ ]:
def batch_iter(X: np.ndarray, y: np.ndarray | None, batch_size: int, shuffle: bool = False, seed: int = 42):
    """Yield mini-batches from NumPy arrays."""
    indices = np.arange(len(X))  # Trial indices.
    if shuffle:  # Shuffle training batches only.
        rng = np.random.default_rng(seed)  # Deterministic shuffle generator.
        rng.shuffle(indices)  # Shuffle in place.
    for start in range(0, len(indices), batch_size):  # Step through mini-batches.
        batch_idx = indices[start:start + batch_size]  # Current batch indices.
        xb = torch.from_numpy(X[batch_idx]).float().to(DEVICE)  # Convert EEG batch to torch tensor.
        if y is None:  # Prediction mode.
            yield xb, None  # Yield inputs only.
        else:
            yb = torch.from_numpy(y[batch_idx]).long().to(DEVICE)  # Convert labels to torch tensor.
            yield xb, yb  # Yield input/label batch.


def run_one_epoch(model: nn.Module, X: np.ndarray, y: np.ndarray, optimizer, criterion, batch_size: int, train: bool, seed: int):
    """Run one training or validation epoch and return loss, accuracy, and F1.

    Important fix:
    when train=True, batches are shuffled. Metrics must therefore compare predictions
    against the batch labels in the same shuffled order, not against the original y array.
    Training updates were already correct; this fixes only the logged train metrics.
    """
    model.train(train)  # Enable dropout/batchnorm training mode only when train=True.
    all_preds = []  # Store predictions in the exact order batches are processed.
    all_targets = []  # Store matching labels in the same batch order as predictions.
    total_loss = 0.0  # Accumulate batch losses.
    n_batches = 0  # Count batches.

    grad_context = torch.enable_grad() if train else torch.no_grad()  # Disable grad in validation.
    with grad_context:  # Enter correct gradient context.
        for xb, yb in batch_iter(X, y, batch_size=batch_size, shuffle=train, seed=seed):  # Iterate batches.
            if train:  # Training step.
                optimizer.zero_grad()  # Clear old gradients.
            logits = model(xb)  # Forward pass.
            loss = criterion(logits, yb)  # Cross-entropy loss.
            if train:  # Backward/update only during training.
                loss.backward()  # Compute gradients.
                optimizer.step()  # Update parameters.
            preds = torch.argmax(logits, dim=1).detach().cpu().numpy()  # Convert predictions to NumPy.
            targets = yb.detach().cpu().numpy()  # Matching labels for this same batch.
            all_preds.append(preds)  # Save predictions.
            all_targets.append(targets)  # Save aligned labels.
            total_loss += float(loss.detach().cpu())  # Accumulate loss.
            n_batches += 1  # Increment batch count.

    preds = np.concatenate(all_preds) if all_preds else np.array([], dtype=int)  # Merge predictions.
    targets = np.concatenate(all_targets) if all_targets else np.array([], dtype=int)  # Merge aligned labels.
    avg_loss = total_loss / max(n_batches, 1)  # Average loss per batch.
    acc = accuracy_score(targets, preds) if len(preds) else 0.0  # Accuracy with aligned labels.
    f1 = f1_score(targets, preds) if len(preds) else 0.0  # Binary F1 with aligned labels.
    return avg_loss, acc, f1, preds  # Return metrics and predictions.


def predict_model(model: nn.Module, X: np.ndarray, batch_size: int) -> np.ndarray:
    """Predict class labels for an array of EEG trials."""
    model.eval()  # Evaluation mode.
    preds = []  # Store batch predictions.
    with torch.no_grad():  # No gradients during prediction.
        for xb, _ in batch_iter(X, None, batch_size=batch_size, shuffle=False):  # Iterate test batches.
            logits = model(xb)  # Forward pass.
            preds.append(torch.argmax(logits, dim=1).detach().cpu().numpy())  # Store predictions.
    return np.concatenate(preds) if preds else np.array([], dtype=int)  # Return all predictions.


def fold_tag(subject_zero_based: int, iteration: int, fold: int) -> str:
    """Stable ID for a subject/iteration/fold checkpoint."""
    return f"B{subject_zero_based + 1:02d}_iter{iteration + 1}_fold{fold + 1}"


def checkpoint_paths(subject_zero_based: int, iteration: int, fold: int) -> dict[str, Path]:
    """Return latest/best checkpoint paths for one fold."""
    tag = fold_tag(subject_zero_based, iteration, fold)
    return {
        "latest": CHECKPOINT_DIR / f"{tag}_latest.pt",
        "best": CHECKPOINT_DIR / f"{tag}_best.pt",
        "progress": CHECKPOINT_DIR / "progress.json",
    }


def save_training_checkpoint(
    model: nn.Module,
    optimizer,
    scheduler,
    subject_zero_based: int,
    iteration: int,
    fold: int,
    epoch: int,
    best_val_acc: float,
    best_val_loss: float,
    best_val_f1: float,
    best_state: dict,
    history: list[dict],
    patience_counter: int,
    epoch_count: int,
    fold_seed: int,
    is_best: bool = False,
) -> None:
    """Save resumable fold state. Overwrites latest checkpoint to avoid unbounded disk growth."""
    if not SAVE_CHECKPOINTS:
        return

    paths = checkpoint_paths(subject_zero_based, iteration, fold)
    payload = {
        "model_state": copy.deepcopy(model.state_dict()),
        "optimizer_state": optimizer.state_dict(),
        "scheduler_state": scheduler.state_dict(),
        "subject_zero_based": subject_zero_based,
        "iteration": iteration,
        "fold": fold,
        "epoch": epoch,  # One-based last completed epoch.
        "best_val_acc": float(best_val_acc),
        "best_val_loss": float(best_val_loss),
        "best_val_f1": float(best_val_f1),
        "best_state": copy.deepcopy(best_state),
        "history": history,
        "patience_counter": int(patience_counter),
        "epoch_count": int(epoch_count),
        "fold_seed": int(fold_seed),
        "run_mode": RUN_MODE,
    }
    torch.save(payload, paths["latest"])
    if is_best:
        torch.save(payload, paths["best"])

    progress = {
        "last_subject": f"B{subject_zero_based + 1:02d}",
        "last_iteration": iteration + 1,
        "last_fold": fold + 1,
        "last_epoch": epoch,
        "best_val_acc": float(best_val_acc),
        "best_val_loss": float(best_val_loss),
        "best_val_f1": float(best_val_f1),
        "latest_checkpoint": str(paths["latest"]),
    }
    with open(paths["progress"], "w") as f:
        json.dump(progress, f, indent=2)


def load_training_checkpoint(model: nn.Module, optimizer, scheduler, subject_zero_based: int, iteration: int, fold: int):
    """Load latest fold checkpoint if resume is enabled and the file exists."""
    if not RESUME_TRAINING:
        return None

    latest_path = checkpoint_paths(subject_zero_based, iteration, fold)["latest"]
    if not latest_path.exists():
        return None

    ckpt = torch.load(latest_path, map_location=DEVICE)
    ckpt_epoch = int(ckpt.get("epoch", 0))
    if ckpt_epoch >= N_EPOCHS:
        print(f"Found completed checkpoint at epoch {ckpt_epoch}/{N_EPOCHS}: {latest_path}")
        print("Not resuming this checkpoint because it would produce an empty training loop. Starting fresh for this fold.")
        return None

    model.load_state_dict(ckpt["model_state"])
    optimizer.load_state_dict(ckpt["optimizer_state"])
    scheduler.load_state_dict(ckpt["scheduler_state"])
    print(f"Resumed fold checkpoint: {latest_path}")
    print(f"Last completed epoch: {ckpt_epoch}")
    return ckpt


def label_counts(y: np.ndarray) -> list[int]:
    """Return class counts with fixed length N_CLASSES."""
    return np.bincount(y.astype(int), minlength=N_CLASSES).astype(int).tolist()


In [ ]:
def train_fold(
    X_train: np.ndarray,
    y_train: np.ndarray,
    X_val: np.ndarray,
    y_val: np.ndarray,
    X_test: np.ndarray,
    y_test: np.ndarray,
    fold_seed: int,
    subject_zero_based: int,
    iteration: int,
    fold: int,
    n_model: int = N_MODEL,
    is_hybrid: bool = IS_HYBRID,
) -> dict:
    """Train one official CV fold and evaluate the fold-best model on the fixed 20% test set."""
    set_all_seeds(fold_seed)  # Make this fold deterministic.
    model = build_model(n_model=n_model, is_hybrid=is_hybrid).to(DEVICE)  # Build fresh model.
    criterion = nn.CrossEntropyLoss()  # Official training objective.
    optimizer = optim.Adam(model.parameters(), lr=LR)  # Official optimizer.
    scheduler = ExponentialLR(optimizer, gamma=GAMMA_LR)  # Official LR schedule object.

    best_val_acc = -np.inf  # Track best validation accuracy, as in the original notebook.
    best_val_f1 = -np.inf  # Track best validation F1 for reporting.
    best_val_loss = np.inf  # Track validation loss for early stopping.
    best_state = copy.deepcopy(model.state_dict())  # Safe default if no epoch improves.
    history = []  # Store per-epoch metrics.
    epoch_count = 0  # Official scheduler steps every STEP_SIZE epochs.
    patience_counter = 0  # Early-stopping counter.
    start_epoch = 0  # Zero-based index of next epoch to run.
    early_stopped = False  # Report whether early stopping fired.

    ckpt = load_training_checkpoint(model, optimizer, scheduler, subject_zero_based, iteration, fold)
    if ckpt is not None:
        start_epoch = int(ckpt["epoch"])
        best_val_acc = float(ckpt.get("best_val_acc", best_val_acc))
        best_val_f1 = float(ckpt.get("best_val_f1", best_val_f1))
        best_val_loss = float(ckpt.get("best_val_loss", best_val_loss))
        best_state = ckpt.get("best_state", copy.deepcopy(model.state_dict()))
        history = list(ckpt.get("history", []))
        patience_counter = int(ckpt.get("patience_counter", 0))
        epoch_count = int(ckpt.get("epoch_count", 0))

    if RUN_SANITY_CHECKS:
        print("Train label counts:", label_counts(y_train))
        print("Val label counts:", label_counts(y_val))
        print("Test label counts:", label_counts(y_test))

    n_train_batches = int(np.ceil(len(X_train) / BATCH_SIZE))
    print(
        f"Training fold now: start_epoch={start_epoch + 1}, target_epoch={N_EPOCHS}, "
        f"train_trials={len(X_train)}, batch_size={BATCH_SIZE}, batches_per_epoch={n_train_batches}"
    )
    if start_epoch >= N_EPOCHS:
        raise RuntimeError(
            "No epochs left to train. This usually means a completed checkpoint was loaded. "
            "Use START_POLICY='fresh_train' with a new RUN_ID, or delete the old output folder."
        )

    for epoch in range(start_epoch, N_EPOCHS):  # Training loop. epoch is zero-based.
        train_loss, train_acc, train_f1, _ = run_one_epoch(  # Train one epoch; metrics are train-mode/dropout-active.
            model, X_train, y_train, optimizer, criterion, BATCH_SIZE, train=True, seed=fold_seed + epoch
        )
        val_loss, val_acc, val_f1, _ = run_one_epoch(  # Validate one epoch.
            model, X_val, y_val, optimizer, criterion, BATCH_SIZE, train=False, seed=fold_seed
        )

        # Optional eval-mode train metrics make train/val comparison fair without doubling every epoch.
        train_eval_loss = np.nan
        train_eval_acc = np.nan
        train_eval_f1 = np.nan
        if TRAIN_EVAL_INTERVAL and ((epoch + 1) == 1 or (epoch + 1) % TRAIN_EVAL_INTERVAL == 0 or (epoch + 1) == N_EPOCHS):
            train_eval_loss, train_eval_acc, train_eval_f1, _ = run_one_epoch(
                model, X_train, y_train, optimizer, criterion, BATCH_SIZE, train=False, seed=fold_seed
            )

        current_lr = optimizer.param_groups[0]["lr"]  # Read current learning rate.
        history.append({  # Save epoch metrics.
            "epoch": epoch + 1,
            "train_loss": train_loss,
            "train_acc": train_acc,
            "train_f1": train_f1,
            "train_eval_loss": train_eval_loss,
            "train_eval_acc": train_eval_acc,
            "train_eval_f1": train_eval_f1,
            "val_loss": val_loss,
            "val_acc": val_acc,
            "val_f1": val_f1,
            "lr": current_lr,
        })

        improved_acc = val_acc > best_val_acc  # Keep original fold-best selection criterion.
        if improved_acc:
            best_val_acc = val_acc  # Update best score.
            best_val_f1 = val_f1  # Validation F1 at best validation accuracy.
            best_state = copy.deepcopy(model.state_dict())  # Save fold-best weights.

        improved_loss = val_loss < (best_val_loss - EARLY_STOP_MIN_DELTA)
        if improved_loss:
            best_val_loss = val_loss  # Keep best validation loss for reporting.

        # Optional early stopping follows the notebook's original fold-selection objective:
        # validation accuracy. A warmup prevents stopping before delayed learning begins.
        if EARLY_STOPPING:
            if improved_acc:
                patience_counter = 0
            elif (epoch + 1) >= EARLY_STOP_WARMUP_EPOCHS:
                patience_counter += 1

        epoch_count += 1  # Increment scheduler counter.
        if epoch_count == STEP_SIZE:  # Official decay condition.
            epoch_count = 0  # Reset counter.
            scheduler.step()  # Decay learning rate.

        save_training_checkpoint(
            model=model,
            optimizer=optimizer,
            scheduler=scheduler,
            subject_zero_based=subject_zero_based,
            iteration=iteration,
            fold=fold,
            epoch=epoch + 1,
            best_val_acc=best_val_acc,
            best_val_loss=best_val_loss,
            best_val_f1=best_val_f1,
            best_state=best_state,
            history=history,
            patience_counter=patience_counter,
            epoch_count=epoch_count,
            fold_seed=fold_seed,
            is_best=improved_acc,
        )

        extra_train_metric = ""
        if not np.isnan(train_eval_acc):
            extra_train_metric = f" | train(eval) acc {train_eval_acc:.4f} f1 {train_eval_f1:.4f}"
        print(  # Compact training log.
            f"Epoch {epoch+1:03d}/{N_EPOCHS} | "
            f"train loss {train_loss:.4f} acc {train_acc:.4f} f1 {train_f1:.4f}{extra_train_metric} | "
            f"val loss {val_loss:.4f} acc {val_acc:.4f} f1 {val_f1:.4f} | lr {current_lr:.6f}"
        )

        if EARLY_STOPPING and patience_counter >= PATIENCE:
            early_stopped = True
            print(f"Early stopping at epoch {epoch + 1}: val loss did not improve for {PATIENCE} epochs.")
            break

    model.load_state_dict(best_state)  # Restore best fold model selected by validation accuracy.
    test_pred = predict_model(model, X_test, batch_size=BATCH_SIZE)  # Evaluate on fixed 20% test set.
    test_acc = accuracy_score(y_test, test_pred)  # Test accuracy.
    test_f1 = f1_score(y_test, test_pred)  # Test F1.
    test_kappa = cohen_kappa_score(y_test, test_pred)  # Test kappa.

    sanity = {}
    if RUN_SANITY_CHECKS:
        sample_n = min(20, len(X_train))
        train_sample_pred = predict_model(model, X_train[:sample_n], batch_size=BATCH_SIZE)
        sanity = {
            "train_label_counts": label_counts(y_train),
            "val_label_counts": label_counts(y_val),
            "test_label_counts": label_counts(y_test),
            "sample_train_labels": y_train[:sample_n].astype(int).tolist(),
            "sample_train_predictions": train_sample_pred.astype(int).tolist(),
        }
        print("Sample train labels:", sanity["sample_train_labels"])
        print("Sample train preds :", sanity["sample_train_predictions"])

    return {  # Return all fold outputs.
        "model_state": best_state,
        "history": history,
        "test_pred": test_pred,
        "test_acc": test_acc,
        "test_f1": test_f1,
        "test_kappa": test_kappa,
        "best_val_acc": best_val_acc,
        "best_val_f1": max(row["val_f1"] for row in history) if history else np.nan,
        "best_val_loss": best_val_loss,
        "epochs_completed": history[-1]["epoch"] if history else 0,
        "early_stopped": early_stopped,
        "sanity": sanity,
    }


### Validation checklist

- **Matches paper/code?** Yes: Adam + CrossEntropy + exponential LR decay every 15 epochs + best validation accuracy selection.
- **Approximation/fix:** Batches are shuffled in this notebook for better training hygiene. The official loop takes contiguous batches without explicit shuffling after split. Set `shuffle=False` in `batch_iter` calls if you want stricter script-level behavior.
- **Data leakage risk?** Test set is never used for fitting model weights inside a fold.
- **Reproducible?** Yes with controlled fold seeds.


## 10. Run one subject with official subject-dependent CV logic


In [ ]:
def run_subject_cv(subject_zero_based: int, iteration: int = 0, n_model: int = N_MODEL, is_hybrid: bool = IS_HYBRID) -> dict:
    """Run the official subject-dependent CV pipeline for one subject and one iteration."""
    X_all, y_all = load_subject_cv_pool(subject_zero_based)  # Merge official T/E sessions.

    # Official 8:2 split after merging all subject sessions. Note: official code does not stratify this split.
    X_data, X_test, y_data, y_test = train_test_split(
        X_all,
        y_all,
        test_size=0.2,
        random_state=42,
    )

    kfold = StratifiedKFold(n_splits=K_FOLDS, shuffle=False)  # Official CV uses shuffle=False.
    fold_outputs = []  # Store fold results.
    subject_dir = OUTPUT_DIR / f"subject_B{subject_zero_based+1:02d}_iter_{iteration+1}"  # Output folder.
    subject_dir.mkdir(parents=True, exist_ok=True)  # Create subject folder.

    for fold, (train_ids, val_ids) in enumerate(kfold.split(X_data, y_data)):  # Iterate official folds.
        print(f"\nSubject B{subject_zero_based+1:02d} | iteration {iteration+1} | fold {fold+1}/{K_FOLDS}")
        fold_seed = BASE_SEED + 1000 * iteration + 100 * subject_zero_based + fold  # Deterministic fold seed.

        fold_result_json = subject_dir / f"fold_{fold+1}_result.json"
        fold_pred_path = subject_dir / f"fold_{fold+1}_test_pred.npy"
        fold_model_path = subject_dir / f"fold_{fold+1}_best_model.pt"
        fold_history_path = subject_dir / f"fold_{fold+1}_history.csv"
        fold_sanity_path = subject_dir / f"fold_{fold+1}_sanity.json"

        if SKIP_COMPLETED_FOLDS and fold_result_json.exists() and fold_pred_path.exists() and fold_model_path.exists():
            print(f"Skipping completed fold {fold+1}; loading saved fold result. Set START_POLICY='fresh_train' and change RUN_ID to force retraining.")
            with open(fold_result_json, "r") as f:
                saved = json.load(f)
            fold_outputs.append({
                "model_state": torch.load(fold_model_path, map_location=DEVICE),
                "history": pd.read_csv(fold_history_path).to_dict("records") if fold_history_path.exists() else [],
                "test_pred": np.load(fold_pred_path),
                "test_acc": saved["test_acc"],
                "test_f1": saved["test_f1"],
                "test_kappa": saved["test_kappa"],
                "best_val_acc": saved["best_val_acc"],
                "best_val_f1": saved["best_val_f1"],
                "best_val_loss": saved.get("best_val_loss", np.nan),
                "epochs_completed": saved.get("epochs_completed", 0),
                "early_stopped": saved.get("early_stopped", False),
                "sanity": saved.get("sanity", {}),
            })
            continue

        fold_out = train_fold(
            X_data[train_ids],
            y_data[train_ids],
            X_data[val_ids],
            y_data[val_ids],
            X_test,
            y_test,
            fold_seed=fold_seed,
            subject_zero_based=subject_zero_based,
            iteration=iteration,
            fold=fold,
            n_model=n_model,
            is_hybrid=is_hybrid,
        )
        fold_outputs.append(fold_out)  # Save fold output.

        history_df = pd.DataFrame(fold_out["history"])  # Convert training history to table.
        history_df.to_csv(fold_history_path, index=False)  # Save history CSV.
        np.save(fold_pred_path, fold_out["test_pred"])  # Save predictions for resume-safe summaries.
        torch.save(fold_out["model_state"], fold_model_path)  # Save best fold model state.

        fold_result = {
            "fold": fold + 1,
            "test_acc": float(fold_out["test_acc"]),
            "test_f1": float(fold_out["test_f1"]),
            "test_kappa": float(fold_out["test_kappa"]),
            "best_val_acc": float(fold_out["best_val_acc"]),
            "best_val_f1": float(fold_out["best_val_f1"]),
            "best_val_loss": float(fold_out["best_val_loss"]),
            "epochs_completed": int(fold_out["epochs_completed"]),
            "early_stopped": bool(fold_out["early_stopped"]),
            "sanity": fold_out.get("sanity", {}),
        }
        with open(fold_result_json, "w") as f:
            json.dump(fold_result, f, indent=2)
        if fold_out.get("sanity"):
            with open(fold_sanity_path, "w") as f:
                json.dump(fold_out["sanity"], f, indent=2)

        plt.figure()  # New figure for accuracy curves.
        plt.plot(history_df["epoch"], history_df["train_acc"], label="train_acc_train_mode")  # Train-mode accuracy.
        if "train_eval_acc" in history_df and history_df["train_eval_acc"].notna().any():
            plt.plot(history_df["epoch"], history_df["train_eval_acc"], label="train_acc_eval_mode")  # Comparable train accuracy.
        plt.plot(history_df["epoch"], history_df["val_acc"], label="val_acc")  # Validation accuracy.
        plt.xlabel("Epoch")  # X-axis label.
        plt.ylabel("Accuracy")  # Y-axis label.
        plt.title(f"B{subject_zero_based+1:02d} fold {fold+1} accuracy")  # Plot title.
        plt.legend()  # Show legend.
        plt.savefig(subject_dir / f"fold_{fold+1}_accuracy.png", dpi=150, bbox_inches="tight")  # Save plot.
        plt.close()  # Close figure to save memory.

        plt.figure()  # New figure for loss curves.
        plt.plot(history_df["epoch"], history_df["train_loss"], label="train_loss_train_mode")  # Train loss.
        if "train_eval_loss" in history_df and history_df["train_eval_loss"].notna().any():
            plt.plot(history_df["epoch"], history_df["train_eval_loss"], label="train_loss_eval_mode")  # Eval-mode train loss.
        plt.plot(history_df["epoch"], history_df["val_loss"], label="val_loss")  # Validation loss.
        plt.xlabel("Epoch")  # X-axis label.
        plt.ylabel("Loss")  # Y-axis label.
        plt.title(f"B{subject_zero_based+1:02d} fold {fold+1} loss")  # Plot title.
        plt.legend()  # Show legend.
        plt.savefig(subject_dir / f"fold_{fold+1}_loss.png", dpi=150, bbox_inches="tight")  # Save plot.
        plt.close()  # Close figure.

        with open(CHECKPOINT_DIR / "progress.json", "w") as f:
            json.dump({
                "last_completed_subject": f"B{subject_zero_based+1:02d}",
                "last_completed_iteration": iteration + 1,
                "last_completed_fold": fold + 1,
                "fold_result_json": str(fold_result_json),
            }, f, indent=2)

    # Official code averages fold test scores for the iteration.
    fold_metrics = pd.DataFrame([
        {
            "fold": i + 1,
            "test_acc": out["test_acc"],
            "test_f1": out["test_f1"],
            "test_kappa": out["test_kappa"],
            "best_val_acc": out["best_val_acc"],
            "best_val_f1": out["best_val_f1"],
            "best_val_loss": out.get("best_val_loss", np.nan),
            "epochs_completed": out.get("epochs_completed", 0),
            "early_stopped": out.get("early_stopped", False),
        }
        for i, out in enumerate(fold_outputs)
    ])
    fold_metrics.to_csv(subject_dir / "fold_metrics.csv", index=False)  # Save fold-level metrics.

    best_fold_idx = int(np.argmax(fold_metrics["test_f1"].values))  # Official code saves model with best fold test F1.
    torch.save(fold_outputs[best_fold_idx]["model_state"], subject_dir / "best_fold_model.pt")  # Save chosen model.

    # Use predictions from the best fold for a concrete confusion matrix artifact.
    best_pred = fold_outputs[best_fold_idx]["test_pred"]  # Best-fold predictions.
    cm = confusion_matrix(y_test, best_pred)  # Confusion matrix for best fold.
    disp = ConfusionMatrixDisplay(cm, display_labels=["Left hand", "Right hand"])  # Build plotter.
    disp.plot(values_format="d")  # Plot confusion matrix.
    plt.title(f"B{subject_zero_based+1:02d} confusion matrix, best fold")  # Title.
    plt.savefig(subject_dir / "confusion_matrix_best_fold.png", dpi=150, bbox_inches="tight")  # Save CM plot.
    plt.close()  # Close figure.

    report = classification_report(y_test, best_pred, target_names=["Left hand", "Right hand"], output_dict=True)  # Report.
    with open(subject_dir / "classification_report_best_fold.json", "w") as f:
        json.dump(report, f, indent=2)  # Save report.

    summary = {  # Iteration summary.
        "subject": f"B{subject_zero_based+1:02d}",
        "iteration": iteration + 1,
        "mean_test_acc_across_folds": float(fold_metrics["test_acc"].mean()),
        "mean_test_f1_across_folds": float(fold_metrics["test_f1"].mean()),
        "mean_test_kappa_across_folds": float(fold_metrics["test_kappa"].mean()),
        "mean_epochs_completed": float(fold_metrics["epochs_completed"].mean()),
        "n_early_stopped_folds": int(fold_metrics["early_stopped"].sum()),
        "best_fold_by_test_f1": best_fold_idx + 1,
        "best_fold_test_acc": float(fold_metrics.loc[best_fold_idx, "test_acc"]),
        "best_fold_test_f1": float(fold_metrics.loc[best_fold_idx, "test_f1"]),
    }
    with open(subject_dir / "summary.json", "w") as f:
        json.dump(summary, f, indent=2)  # Save iteration summary.

    print("\nSubject summary:")  # Print summary header.
    print(json.dumps(summary, indent=2))  # Print summary JSON.
    return {"summary": summary, "fold_metrics": fold_metrics, "subject_dir": str(subject_dir)}  # Return outputs.


In [ ]:
# Run configured subjects only when dataset files exist.
all_subject_results = []  # Collect subject-level results.

if len(missing_files) == 0:
    for iteration in range(N_ITER):  # Repeat runs.
        for subject in SUBJECTS:  # Iterate configured subjects.
            result = run_subject_cv(subject, iteration=iteration, n_model=N_MODEL, is_hybrid=IS_HYBRID)  # Run one subject.
            all_subject_results.append(result["summary"])  # Collect summary.
            with open(OUTPUT_DIR / "all_subject_iteration_summaries_partial.json", "w") as f:
                json.dump(all_subject_results, f, indent=2)  # Save partial aggregate after each subject.
else:
    print("Cannot run training because MATLAB dataset files are missing.")
    print("Place B01T.mat/B01E.mat ... B09T.mat/B09E.mat under DATA_DIR, then rerun from Section 5.")



Subject B01 | iteration 1 | fold 1/5
Train label counts: [233, 227]
Val label counts: [59, 57]
Test label counts: [68, 76]
Training fold now: start_epoch=1, target_epoch=150, train_trials=460, batch_size=128, batches_per_epoch=4
Epoch 001/150 | train loss 0.7450 acc 0.5196 f1 0.2801 | train(eval) acc 0.5065 f1 0.0000 | val loss 0.6920 acc 0.5086 f1 0.0000 | lr 0.001000
Epoch 002/150 | train loss 0.7023 acc 0.5022 f1 0.2730 | val loss 0.6921 acc 0.5345 f1 0.5846 | lr 0.001000
Epoch 003/150 | train loss 0.6907 acc 0.5217 f1 0.1339 | val loss 0.6951 acc 0.5172 f1 0.6543 | lr 0.001000
Epoch 004/150 | train loss 0.6935 acc 0.5130 f1 0.0588 | val loss 0.6952 acc 0.5000 f1 0.6375 | lr 0.001000
Epoch 005/150 | train loss 0.6933 acc 0.5087 f1 0.0424 | val loss 0.6942 acc 0.4569 f1 0.4706 | lr 0.001000
Epoch 006/150 | train loss 0.6925 acc 0.5130 f1 0.0261 | val loss 0.6936 acc 0.4828 f1 0.2683 | lr 0.001000
Epoch 007/150 | train loss 0.6933 acc 0.5087 f1 0.0088 | val loss 0.6934 acc 0.5086 f1 

### Validation checklist

- **Matches paper/code?** Yes for subject-dependent CV split and fold structure.
- **Approximation?** Debug mode uses fewer epochs/folds and must not be compared to paper results.
- **Data leakage risk?** The fixed 20% test set is used after each fold, matching official code. However, selecting the saved model by test F1 is not ideal methodology; the notebook reports fold means as the official metric.
- **Reproducible?** Yes.


## 11. Aggregate results and compare with paper targets


In [ ]:
PAPER_TARGETS = {  # Paper-reported subject-dependent CV targets.
    "Hybrid_EEGNet_VQC1_acc": 0.8382,  # 83.82% accuracy from paper Table 5 for Hybrid EEGNet.
    "Hybrid_EEGNet_VQC1_f1": 0.8388,  # Table 4 average F1 for Hybrid EEGNet.
    "Hybrid_DCN_VQC1_acc": np.nan,  # The paper does not report a separate Table-5 accuracy for Hybrid DCN.
    "Hybrid_DCN_VQC1_f1": 0.8062,  # Table 4 average F1 for Hybrid Deep ConvNet.
}

if all_subject_results:  # Build aggregate table only if training ran.
    results_df = pd.DataFrame(all_subject_results)  # Convert summaries to table.
    results_df.to_csv(OUTPUT_DIR / "all_subject_iteration_summaries.csv", index=False)  # Save table.
    display(results_df)  # Display raw summaries.

    paper_acc = PAPER_TARGETS["Hybrid_EEGNet_VQC1_acc"] if N_MODEL == 1 else PAPER_TARGETS["Hybrid_DCN_VQC1_acc"]  # Accuracy target.
    paper_f1 = PAPER_TARGETS["Hybrid_EEGNet_VQC1_f1"] if N_MODEL == 1 else PAPER_TARGETS["Hybrid_DCN_VQC1_f1"]  # F1 target.

    aggregate = {  # Aggregate across configured subjects/iterations.
        "model": "Hybrid EEGNet + VQC1" if (N_MODEL == 1 and IS_HYBRID) else "Configured model",
        "debug_mode": DEBUG_MODE,
        "n_subjects": len(SUBJECTS),
        "n_iter": N_ITER,
        "mean_acc": float(results_df["mean_test_acc_across_folds"].mean()),
        "std_acc": float(results_df["mean_test_acc_across_folds"].std(ddof=0)),
        "mean_f1": float(results_df["mean_test_f1_across_folds"].mean()),
        "std_f1": float(results_df["mean_test_f1_across_folds"].std(ddof=0)),
        "paper_acc_target": paper_acc,
        "paper_f1_target": paper_f1,
    }
    aggregate["acc_gap_vs_paper"] = aggregate["mean_acc"] - paper_acc if not np.isnan(paper_acc) else np.nan  # Difference from paper.
    aggregate["f1_gap_vs_paper"] = aggregate["mean_f1"] - paper_f1  # Difference from paper.

    comparison_df = pd.DataFrame([aggregate])  # One-row comparison table.
    comparison_df.to_csv(OUTPUT_DIR / "paper_comparison.csv", index=False)  # Save comparison.
    display(comparison_df)  # Show comparison.
else:
    print("No computed metrics yet. After running the training section, comparison tables will be saved here:")
    print(OUTPUT_DIR)


,subject,iteration,mean_test_acc_across_folds,mean_test_f1_across_folds,mean_test_kappa_across_folds,mean_epochs_completed,n_early_stopped_folds,best_fold_by_test_f1,best_fold_test_acc,best_fold_test_f1
0,B01,1,0.726389,0.760263,0.44533,150.0,0,1,0.784722,0.824859


,model,debug_mode,n_subjects,n_iter,mean_acc,std_acc,mean_f1,std_f1,paper_acc_target,paper_f1_target,acc_gap_vs_paper,f1_gap_vs_paper
0,Hybrid EEGNet + VQC1,False,1,1,0.726389,0.0,0.760263,0.0,0.8382,0.8388,-0.111811,-0.078537


### Validation checklist

- **Matches paper/code?** The comparison uses paper-reported CV targets for Hybrid EEGNet/DeepConvNet + VQC1.
- **Approximation?** Debug results are not reproduction evidence.
- **Data leakage risk?** None in metric aggregation.
- **Reproducible?** Yes.


## 12. Optional: run the authors' original scripts inside Kaggle

Use this only if you uploaded or added `capsule-6063621-code.zip` to the Kaggle notebook and want a direct script-level comparison. The self-contained implementation above is already aligned to the official PyTorch CV code path.


In [ ]:
# Optional original-capsule execution helper for Kaggle.
# Put capsule-6063621-code.zip in /kaggle/input or upload it to the notebook session first.

# !mkdir -p /kaggle/working/capsule_code
# !unzip -q /kaggle/input/<your-capsule-dataset>/capsule-6063621-code.zip -d /kaggle/working/capsule_code
# %cd /kaggle/working/capsule_code
# !python py_eeg_classifier.py \
#     --dataset 1 \
#     --dataset_path /kaggle/input/datasets/tousifbnn/bci-iv-2b/ \
#     --results_path /kaggle/working/original_capsule_results/cv \
#     --n_iter 1 \
#     --sub_list 0 \
#     --n_epochs 2 \
#     --batch_size 16 \
#     --n_model 1 \
#     --isHybrid true \
#     --cross_val true \
#     --kernel_length 125 \
#     --pool_sz 5 \
#     --max_pool true \
#     --n_qubits 4 \
#     --q_depth 6 \
#     --q_delta 0.01 \
#     --ansatz 0


### Validation checklist

- **Matches paper/code?** This directly invokes the official script when the full data environment exists.
- **Approximation?** The example command uses debug epochs/batch size; increase to paper settings for final reproduction.
- **Data leakage risk?** Same as official script.
- **Reproducible?** Depends on the unmodified capsule and data package.


## 13. Known weaknesses to document in any reproduction report

1. **The official CV preprocessing is not fold-local.** The scaler is fit on the original T-session train file before T/E are merged and split for CV. This matches the code but is not the cleanest possible CV hygiene.
2. **The paper text and script default differ on CV batch size.** Paper text says 128; parser default is 64. Report which one you use.
3. **The official code saves the fold model with best test F1.** That is not a valid model-selection rule. Use fold-mean test metrics for reporting; do not claim a best-fold-selected number as unbiased performance.
4. **Quantum simulation is slow.** Full B1–B9 × 5 iterations × 5 folds × 300 epochs is expensive in Colab.
5. **No quantum advantage is shown.** The paper itself concludes that the QNN behaves similarly to a fully connected classifier for this task.

For our project, the strongest immediate reproduction target is still **Hybrid EEGNet + VQC1 subject-dependent CV**, then **Hybrid DeepConvNet + VQC1**, then the more complex VQC2/ATCNet hold-out path.


## 14. Strict package + download outputs

Run this only after training. It refuses to package the old `official_aligned` folder by mistake and creates a run-specific zip from the active `OUTPUT_DIR`.


In [ ]:
# ============================================================
# Safe browser download cell — avoids SameFileError
# ============================================================

from pathlib import Path
import shutil
from IPython.display import display, FileLink
import time

# created_zip already exists from the export cell above
created_zip = Path(created_zip)

# Put/download from /content
local_zip_path = Path("/content") / created_zip.name

# Avoid copying file onto itself
if created_zip.resolve() != local_zip_path.resolve():
    shutil.copy2(created_zip, local_zip_path)
    print(f"Copied zip locally: {local_zip_path}")
else:
    print(f"Zip already local: {local_zip_path}")

print("\nIf auto-download does not start, click this link:")
display(FileLink(str(local_zip_path)))

try:
    from google.colab import files
    print("\nTrying auto-download now...")
    time.sleep(1)
    files.download(str(local_zip_path))
except Exception as e:
    print("\nAuto-download failed or was blocked.")
    print("Use the clickable link above.")
    print("Error:", repr(e))

Active RUN_ID: metricfix_v5_b01_150ep
Active START_POLICY: fresh_train
Active OUTPUT_DIR: /content/eeg_qml_olvera_metricfix_v5_b01_150ep_outputs
Expected output folder: eeg_qml_olvera_metricfix_v5_b01_150ep_outputs

Checking: subject_B01_iter_1
  fold 1: rows=150, max_epoch=150/150, status=complete
  fold 2: rows=150, max_epoch=150/150, status=complete
  fold 3: rows=150, max_epoch=150/150, status=complete
  fold 4: rows=150, max_epoch=150/150, status=complete
  fold 5: rows=150, max_epoch=150/150, status=complete

Zip roots: ['eeg_qml_olvera_metricfix_v5_b01_150ep_outputs']

Created run-specific zip: /content/eeg_qml_olvera_metricfix_v5_b01_150ep_outputs.zip
Created legacy alias:    /content/eeg_qml_olvera_outputs.zip
Zip size: 1.41 MB


SameFileError: PosixPath('/content/eeg_qml_olvera_metricfix_v5_b01_150ep_outputs.zip') and PosixPath('/content/eeg_qml_olvera_metricfix_v5_b01_150ep_outputs.zip') are the same file